Model Packaging & Reproducibility


Setup

In [1]:
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reusable preprocessing pipeline (heart_pipeline.py)

In [2]:
%%writefile heart_pipeline.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAW_COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
               "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
BINARY = ["sex", "fbs", "exang"]
CATEGORICAL = ["cp", "restecg", "slope", "thal"]
FEATURES = NUMERIC + BINARY + CATEGORICAL
RANDOM_STATE = 42


def clean_raw(path):
    """Raw UCI file -> clean dataframe with binary target (same steps as Task 1)."""
    df = pd.read_csv(path, header=None, names=RAW_COLUMNS, na_values="?")
    df["target"] = (df["num"] > 0).astype(int)
    df = df.drop(columns="num")
    for col in ["ca", "thal"]:
        df[col] = df[col].fillna(df[col].mode()[0])
    return df.drop_duplicates().reset_index(drop=True)


def build_preprocessor():
    """Scaling for numbers, passthrough for 0/1 columns, one-hot for categories."""
    return ColumnTransformer([
        ("scale", StandardScaler(), NUMERIC),
        ("keep", "passthrough", BINARY),
        ("onehot", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])


def build_final_model():
    """The selected model from Task 2 with its tuned settings."""
    return Pipeline([
        ("prep", build_preprocessor()),
        ("model", LogisticRegression(C=1, class_weight="balanced", max_iter=1000)),
    ])

Writing heart_pipeline.py


Train final model using the reusable pipeline

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from heart_pipeline import FEATURES, RANDOM_STATE, build_final_model

df = pd.read_csv("heart_clean.csv")
X, y = df[FEATURES], df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

final_model = build_final_model().fit(X_train, y_train)
print("Test accuracy:", round(final_model.score(X_test, y_test), 3))

Test accuracy: 0.869


Save the model (joblib + MLflow format)

In [4]:
import joblib, json, sklearn, mlflow.sklearn, shutil, os
from datetime import datetime
from mlflow.models import infer_signature

os.makedirs("model", exist_ok=True)

# Format 1: joblib (pickle) - one file with preprocessing + model
joblib.dump(final_model, "model/heart_model.joblib")

# Format 2: MLflow model folder - includes the environment and input format
shutil.rmtree("model/mlflow_model", ignore_errors=True)
mlflow.sklearn.save_model(
    final_model, "model/mlflow_model",
    signature=infer_signature(X_train, final_model.predict(X_train)),
    input_example=X_train.head(3),
    skops_trusted_types=["sklearn.compose._column_transformer._RemainderColsList"],
)

metadata = {
    "model": "Logistic Regression (C=1, class_weight=balanced)",
    "features": FEATURES,
    "test_accuracy": round(final_model.score(X_test, y_test), 4),
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "trained_on": datetime.now().strftime("%Y-%m-%d %H:%M"),
}
json.dump(metadata, open("model/model_metadata.json", "w"), indent=2)
print(json.dumps(metadata, indent=2))
!ls -R model

{
  "model": "Logistic Regression (C=1, class_weight=balanced)",
  "features": [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
    "ca",
    "sex",
    "fbs",
    "exang",
    "cp",
    "restecg",
    "slope",
    "thal"
  ],
  "test_accuracy": 0.8689,
  "random_state": 42,
  "sklearn_version": "1.6.1",
  "trained_on": "2026-10-20 05:10"
}
model:
heart_model.joblib  mlflow_model  model_metadata.json

model/mlflow_model:
conda.yaml	    MLmodel	 python_env.yaml   serving_input_example.json
input_example.json  model.skops  requirements.txt


Reproducibility check

In [7]:
import numpy as np

reloaded_joblib = joblib.load("model/heart_model.joblib")
reloaded_mlflow = mlflow.sklearn.load_model("model/mlflow_model")
task2_model = joblib.load("heart_model.joblib")       # the model saved in Task 2

p_new = final_model.predict_proba(X_test)[:, 1]
print("joblib reload identical :", np.allclose(p_new, reloaded_joblib.predict_proba(X_test)[:, 1]))
print("MLflow reload identical :", np.allclose(p_new, reloaded_mlflow.predict_proba(X_test)[:, 1]))
print("Same as Task 2 model    :", np.allclose(p_new, task2_model.predict_proba(X_test)[:, 1]))

# Use the packaged model on one new patient
patient = pd.DataFrame([{"age": 58, "sex": 1, "cp": 4, "trestbps": 140, "chol": 289, "fbs": 0,
                         "restecg": 2, "thalach": 122, "exang": 1, "oldpeak": 2.4,
                         "slope": 2, "ca": 2, "thal": 7}])
prob = reloaded_joblib.predict_proba(patient)[0, 1]
print(f"\nNew patient -> disease probability {prob:.2f}, prediction {int(prob >= 0.5)}")

joblib reload identical : True
MLflow reload identical : True
Same as Task 2 model    : True

New patient -> disease probability 1.00, prediction 1


requirements.txt (exact package versions)

In [8]:
from importlib.metadata import version
import sys

packages = ["pandas", "numpy", "scikit-learn", "joblib", "mlflow", "matplotlib", "seaborn"]
lines = [f"# Python {sys.version.split()[0]}"] + [f"{p}=={version(p)}" for p in packages]
open("requirements.txt", "w").write("\n".join(lines) + "\n")
print(open("requirements.txt").read())

# Python 3.13.16
pandas==2.2.3
numpy==2.1.3
scikit-learn==1.6.1
joblib==1.6.0
mlflow==3.17.1
matplotlib==3.10.0
seaborn==0.13.2



Download packaged model

In [9]:
!zip -qr task4_package.zip model heart_pipeline.py requirements.txt
from google.colab import files
files.download("task4_package.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

- Final model: Logistic Regression (C=1, class_weight=balanced), trained from the
  reusable pipeline file heart_pipeline.py (cleaning, scaling, one-hot encoding + model).
- Saved in two formats: joblib (model/heart_model.joblib) and MLflow model format
  (model/mlflow_model, which records the environment and input signature),
  plus model_metadata.json.
- Reproducibility check: reloading both formats and retraining from the pipeline file
  gave identical predictions to the Task 2 model (all checks True). Fixed
  random_state=42 for the split and model.
- requirements.txt pins exact package versions (Python 3.13, scikit-learn 1.6.1).